# Seizure Simulation in Clustered Networks — NEURON

A single-compartment **Hodgkin–Huxley + A-current** clustered network that
produces spontaneous **network bursts** under weak Poisson drive. The A-current
density `gbar_kA` is the **4-AP knob**: partially reducing it raises the burst
frequency; a strong block merges bursts into continuous firing.

This notebook mirrors the LIF project's simulation notebook. It will:

1. build a log-normal clustered topology (the preferred, realistic builder),
2. **verify the network bursts** in the normal state (a gate before dataset generation),
3. compare **normal vs 4-AP**,
4. trace a **dose-response** curve (burst frequency vs `gbar_kA`),
5. **save an inference-ready dataset**, and
6. run the **CCG + learned-LIF inference** on the generated data (AUC / FDR).

> **Prerequisite:** compile the mechanisms once before running:
> ```bash
> cd neuron_simulation && nrnivmodl mechanisms
> ```
> (`build_network` loads them automatically.)

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

# Make the project packages importable from the notebooks/ folder.
REPO_ROOT = os.path.abspath('..')
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'inference')):
    if p not in sys.path:
        sys.path.insert(0, p)

from neuron_simulation import (
    topology, build_network, run_simulation, states, analysis, plotting, workflows,
)

# --- Configuration (the validated bursting regime) ---
CONFIG = {
    'topology': dict(
        num_clusters=14, neurons_per_cluster_range=(8, 12),
        inhibitory_probability=0.2, space_size=13.0,
        target_density=0.04, ln_sigma=1.0, seed=1,
    ),
    # Weak noise = ignition seed; strong recurrence = propagation;
    # A-current + depression = termination.
    'build': dict(
        noise_weight=0.0006, noise_rate=20.0, exc_weight_scale=3.0,
        gbar_kA_exc=0.006, gbar_kA_inh=0.004,
        depression=True, depression_d=0.5, tau_d=800.0,
        celsius=6.3,
    ),
    'sim': dict(dt=0.025, discard_transient_ms=1000.0),
}
print('config ready')

## 1. Build the topology (log-normal — preferred)

The log-normal builder gives a continuous, heavy-tailed degree distribution
(no bimodal gap) at a realistic sparse density (~3–4%). Switch to
`topology.build_topology(...)` for the discrete-hub variant (denser, bimodal).

In [ ]:
topo = topology.build_topology_lognormal(**CONFIG['topology'])
N = topo['n_neurons']
print(f"N = {N}, synapses = {len(topo['connections'])}, "
      f"density = {topo['cluster_info']['density']*100:.2f}%")

In [ ]:
# Spatial map + degree distribution (log-normal is heavy-tailed, not bimodal).
fig1 = plotting.plot_topology_map(
    topo['neuron_positions'], topo['connections'],
    topo['cluster_assignments'], topo['neuron_is_inhibitory'])
fig2 = plotting.plot_degree_distribution(
    topo['connections'], N,
    connection_propensity=topo['cluster_info'].get('connection_propensity'))
plt.show()

## 2. Verify the network bursts (normal state)

**Gate:** confirm discrete, high-participation network bursts *before* generating
any dataset. A network burst = **> 80% of neurons firing within the event window**
(measured post burn-in).

In [ ]:
normal = workflows.run_single_state(
    topo, state=states.normal_state(), build_kwargs=CONFIG['build'],
    duration=8000.0, **CONFIG['sim'])
stats = normal['burst_stats']
print(stats)
assert stats['n_bursts'] >= 3 and not stats['merged'], 'network did not burst — retune before proceeding'
assert stats['mean_participation'] >= 0.8, 'bursts below 80% participation'
print('OK: network bursts with >=80% participation')

In [ ]:
fig = plotting.plot_raster(
    normal['spike_data'], N, 8000.0,
    is_inhibitory=topo['neuron_is_inhibitory'],
    cluster_assignments=topo['cluster_assignments'], burn_in_ms=0.0,
    title='Normal state (gbar_kA = 0.006)')
plt.show()

## 3. Normal vs 4-AP

4-AP is a **partial** reduction of `gbar_kA`. The weakened brake should make
bursts **more frequent** (shorter inter-burst intervals).

In [ ]:
four_ap = workflows.run_single_state(
    topo, state=states.four_ap_state(block_fraction=0.2), build_kwargs=CONFIG['build'],
    duration=8000.0, **CONFIG['sim'])
print('normal burst rate :', round(normal['burst_stats']['burst_rate_hz'], 2), 'Hz')
print('4-AP   burst rate :', round(four_ap['burst_stats']['burst_rate_hz'], 2), 'Hz')

fig = plotting.plot_state_comparison(
    (normal['spike_data'], 'Normal (gbar_kA=0.006)'),
    (four_ap['spike_data'], '4-AP partial block (gbar_kA=0.0048)'),
    N, 8000.0, burn_in_ms=0.0)
plt.show()

## 4. Dose-response: burst frequency vs `gbar_kA`

Sweep the A-current density from drug-free toward a strong block. Expect the
burst rate to **rise** through the partial-block window and then **merge** into
continuous firing at strong block (hollow markers).

In [ ]:
sweep = states.dose_response_gbar(n_points=6, min_fraction=1.0, max_fraction=0.35)
gbars, rates, merged_flags = [], [], []
for s in sweep:
    res = workflows.run_single_state(
        topo, state=s, build_kwargs=CONFIG['build'], duration=5000.0, **CONFIG['sim'])
    gbars.append(s['gbar_kA_exc'])
    rates.append(res['burst_stats']['burst_rate_hz'])
    merged_flags.append(res['burst_stats']['merged'])
    print(f"gbar={s['gbar_kA_exc']:.4f} -> {rates[-1]:.2f} Hz, merged={merged_flags[-1]}")

fig = plotting.plot_burst_frequency_curve(gbars, rates, merged_flags=merged_flags)
plt.show()

## 5. Generate an inference-ready dataset

Save several independent recordings (same wired network, different noise seeds)
in the exact LIF session layout. Set `record_voltage=True` to also enable the
voltage-augmented inference mode.

In [ ]:
meta, session_dir = workflows.generate_dataset(
    n_recordings=3, recording_duration=20000.0,
    topology_kind='lognormal', topology_kwargs=CONFIG['topology'],
    build_kwargs=CONFIG['build'], state=states.normal_state(),
    record_voltage=False, save_dir='NEURON data', **CONFIG['sim'])
print('session saved to:', session_dir)

## 6. Validate the format and run inference

First assert the saved session matches the inference input contract (fields,
shapes, dtypes), then run the **CCG baseline + learned-LIF** model and report
**AUC / FDR** against the ground-truth wiring — exactly as for the LIF data.

In [ ]:
import adapter  # inference/adapter.py
adapter.validate_session_format(session_dir)

# Learned-LIF trains a small torch model; keep epochs modest for the demo.
summary = adapter.run_inference(
    session_dir, run_learned=True, run_ccg=True,
    learned_params={'n_epochs': 30, 'K': 80})
print(summary)

## Notes & caveats

- **Burst rate is fast** (~2–3 Hz) vs real cultures (~0.03 Hz). Convenient for
  generating many bursts quickly, but not culture-realistic timing.
- **Squid HH kinetics** at 6.3 °C; set `celsius=34` in `CONFIG['build']` for a
  faster mammalian-like variant.
- **Sparse log-normal topology gives a messier 4-AP effect** than a dense
  discrete-hub network. Use `topology.build_topology(...)` for the sharpest
  4-AP contrast (but a bimodal, less realistic degree distribution).
- The **ground truth is the exact wired graph** (`connections`); inference sees
  steady-state data (the first ~1 s transient is dropped at save time).